In [4]:
!pip install presidio-analyzer presidio-anonymizer promptlayer openai

In [14]:
# ============================================================
# LLM MIDDLEWARE PIPELINE — LangSmith Tracing
# Google Colab Ready
# ============================================================

# -----------------------------
# CELL 1 — INSTALL DEPENDENCIES
# -----------------------------

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("presidio-analyzer")
install("presidio-anonymizer")
install("litellm")
install("langsmith")
install("openai")
install("spacy")

subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_lg"])


# -----------------------------
# CELL 2 — API KEYS
# -----------------------------

import os

os.environ["LANGCHAIN_API_KEY"] = "YOUR_LANGSMITH_KEY"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "llm-demo-pipeline"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"


# -----------------------------
# CELL 3 — IMPORTS
# -----------------------------

import re
import zlib
import hashlib
import uuid
from datetime import datetime, timezone

from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

from langsmith import traceable, Client as LangSmithClient

import litellm
litellm.success_callback = ["langsmith"]
litellm.set_verbose = False

ls_client = LangSmithClient()


# -----------------------------
# CELL 4 — PRESIDIO SETUP
# -----------------------------

provider = NlpEngineProvider(
    nlp_configuration={
        "nlp_engine_name": "spacy",
        "models": [{"lang_code": "en", "model_name": "en_core_web_lg"}],
    }
)

analyzer = AnalyzerEngine(nlp_engine=provider.create_engine())
anonymizer = AnonymizerEngine()

PII_ENTITIES = [
    "PERSON",
    "EMAIL_ADDRESS",
    "PHONE_NUMBER",
    "CREDIT_CARD",
    "US_SSN",
    "IP_ADDRESS",
    "LOCATION",
    "DATE_TIME",
    "URL",
]

ANONYMIZER_OPERATORS = {
    "PERSON": OperatorConfig("replace", {"new_value": "<PERSON>"}),
    "EMAIL_ADDRESS": OperatorConfig(
        "mask",
        {"type": "mask", "masking_char": "*", "chars_to_mask": 6, "from_end": False},
    ),
    "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "<PHONE>"}),
    "CREDIT_CARD": OperatorConfig("replace", {"new_value": "<CREDIT_CARD>"}),
    "US_SSN": OperatorConfig("hash", {"hash_type": "sha256"}),
    "IP_ADDRESS": OperatorConfig("replace", {"new_value": "<IP_ADDRESS>"}),
    "LOCATION": OperatorConfig("replace", {"new_value": "<LOCATION>"}),
    "DATE_TIME": OperatorConfig("replace", {"new_value": "<DATE>"}),
    "URL": OperatorConfig("redact", {}),
}


# -----------------------------
# CELL 5 — PII DETECTION
# -----------------------------

@traceable(name="PII Detection", tags=["presidio", "pii"])
def detect_pii(text: str) -> dict:

    results = analyzer.analyze(text=text, entities=PII_ENTITIES, language="en")

    detected = [
        {
            "entity_type": r.entity_type,
            "value": text[r.start:r.end],
            "confidence": round(r.score, 2),
            "start": r.start,
            "end": r.end,
        }
        for r in results
    ]

    return {
        "raw_results": results,
        "detected_summary": detected,
        "total_found": len(detected),
    }


# -----------------------------
# CELL 6 — ANONYMIZATION
# -----------------------------

@traceable(name="Anonymization", tags=["presidio", "anonymization"])
def anonymize_text(text: str, pii_raw_results: list) -> dict:

    if not pii_raw_results:
        return {"original": text, "anonymized": text, "changes": 0}

    result = anonymizer.anonymize(
        text=text,
        analyzer_results=pii_raw_results,
        operators=ANONYMIZER_OPERATORS,
    )

    return {
        "original": text,
        "anonymized": result.text,
        "items": [str(i) for i in result.items],
        "changes": len(result.items),
    }


# -----------------------------
# CELL 7 — COMPRESSION
# -----------------------------

FILLER_WORDS = [
    r"\bplease\b",
    r"\bkindly\b",
    r"\bjust\b",
    r"\bbasically\b",
    r"\bactually\b",
    r"\bvery\b",
    r"\breally\b",
    r"\bsimply\b",
]


@traceable(name="Prompt Compression", tags=["compression"])
def compress_prompt(text: str) -> dict:

    cleaned = text
    for filler in FILLER_WORDS:
        cleaned = re.sub(filler, "", cleaned, flags=re.IGNORECASE)

    cleaned = re.sub(r"\s{2,}", " ", cleaned).strip()

    compressed = zlib.compress(cleaned.encode("utf-8"), level=9)
    compressed_hex = compressed.hex()

    return {
        "compressed_hex": compressed_hex,
        "original_chars": len(text),
        "cleaned_chars": len(cleaned),
        "compressed_bytes": len(compressed),
        "compression_ratio": round(len(compressed) / max(len(text.encode()), 1), 3),
    }


def decompress_prompt(compressed_hex: str) -> str:
    return zlib.decompress(bytes.fromhex(compressed_hex)).decode("utf-8")


# -----------------------------
# CELL 8 — PROMPT VERSIONING
# -----------------------------

_local_version_store = {}


@traceable(name="Prompt Versioning", tags=["versioning"])
def save_prompt_version(prompt: str, prompt_name: str, metadata: dict = None) -> dict:

    content_hash = hashlib.sha256(prompt.encode()).hexdigest()[:12]

    if content_hash in _local_version_store:
        existing = _local_version_store[content_hash]
        return {"version_id": existing["version_id"], "status": "already_exists"}

    version_id = f"v_{uuid.uuid4().hex[:8]}"

    try:
        ls_client.push_prompt(
            prompt_name,
            object={"messages": [{"role": "user", "content": prompt}]},
            description=f"Pipeline version {version_id}",
            tags=["middleware-pipeline"],
        )
        hub_status = "pushed_to_langsmith_hub"
    except Exception as e:
        hub_status = str(e)

    record = {
        "version_id": version_id,
        "prompt_name": prompt_name,
        "content_hash": content_hash,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "metadata": metadata or {},
        "hub_status": hub_status,
    }

    _local_version_store[content_hash] = record

    return {"version_id": version_id, "status": hub_status}


def list_versions():
    return list(_local_version_store.values())


# -----------------------------
# CELL 9 — LLM CALL
# -----------------------------

@traceable(name="LLM Call", tags=["litellm"])
def call_llm(prompt: str, model: str = "gpt-4o-mini") -> dict:

    response = litellm.completion(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=1000,
    )

    return {
        "reply": response.choices[0].message.content,
        "model": model,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
    }


# -----------------------------
# CELL 10 — PIPELINE
# -----------------------------

@traceable(name="LLM Middleware Pipeline", tags=["pipeline"])
def run_pipeline(
    raw_prompt: str,
    prompt_name: str = "middleware_prompt",
    llm_model: str = "gpt-4o-mini",
    send_to_llm: bool = False,
) -> dict:

    session_id = uuid.uuid4().hex[:10]

    pii_output = detect_pii(raw_prompt)

    anon_output = anonymize_text(raw_prompt, pii_output["raw_results"])

    comp_output = compress_prompt(anon_output["anonymized"])

    version_output = save_prompt_version(
        prompt=anon_output["anonymized"],
        prompt_name=prompt_name,
        metadata={"session_id": session_id, "model": llm_model},
    )

    llm_output = None
    if send_to_llm:
        safe_prompt = decompress_prompt(comp_output["compressed_hex"])
        llm_output = call_llm(safe_prompt, model=llm_model)

    return {
        "session_id": session_id,
        "version_id": version_output["version_id"],
        "safe_prompt": anon_output["anonymized"],
        "pii_detected": pii_output["detected_summary"],
        "compression_stats": {
            k: v for k, v in comp_output.items() if k != "compressed_hex"
        },
        "versioning": version_output,
        "llm_response": llm_output,
    }


# -----------------------------
# CELL 11 — DEMO
# -----------------------------

sample_prompt = """
Please help me draft a professional email. My name is John Smith and I work at Acme Corp.
My email is john.smith@acme.com and my phone is +1-800-555-0199.
I am trying to reach Sarah Johnson at sarah.j@gmail.com regarding an invoice.
Her SSN is 123-45-6789 and her IP address is 192.168.1.42.
The meeting is scheduled for 03/15/2024 in New York.
"""

result = run_pipeline(
    raw_prompt=sample_prompt,
    prompt_name="email_draft_v1",
    llm_model="gpt-4o-mini",
    send_to_llm=False,
)

print("Session:", result["session_id"])
print("Version:", result["version_id"])
print("PII:", result["pii_detected"])
print("Safe Prompt:", result["safe_prompt"])
print("Compression:", result["compression_stats"])

Session: 11377259f6
Version: v_8eec7b8a
PII: [{'entity_type': 'EMAIL_ADDRESS', 'value': 'john.smith@acme.com', 'confidence': 1.0, 'start': 103, 'end': 122}, {'entity_type': 'EMAIL_ADDRESS', 'value': 'sarah.j@gmail.com', 'confidence': 1.0, 'start': 194, 'end': 211}, {'entity_type': 'IP_ADDRESS', 'value': '192.168.1.42', 'confidence': 0.95, 'start': 279, 'end': 291}, {'entity_type': 'PERSON', 'value': 'John Smith', 'confidence': 0.85, 'start': 55, 'end': 65}, {'entity_type': 'PERSON', 'value': 'Sarah Johnson', 'confidence': 0.85, 'start': 177, 'end': 190}, {'entity_type': 'LOCATION', 'value': 'New York', 'confidence': 0.85, 'start': 336, 'end': 344}, {'entity_type': 'PHONE_NUMBER', 'value': '+1-800-555-0199', 'confidence': 0.75, 'start': 139, 'end': 154}, {'entity_type': 'DATE_TIME', 'value': '03/15/2024', 'confidence': 0.6, 'start': 322, 'end': 332}, {'entity_type': 'URL', 'value': 'john.sm', 'confidence': 0.5, 'start': 103, 'end': 110}, {'entity_type': 'URL', 'value': 'acme.com', 'conf